# 05 Path diagrams: one carry and one no-carry prompt

36+59=95 (carry) and 36+53=89 (no carry): same a, same tens of b, only b's units differ, which flips the carry. Both are answered correctly in the clean run.

For each prompt: the attribution graph, pruned as in circuit-tracer's viewer, saved as small CSVs in `results/path_diagram/<a>_<b>/` (nodes, edges); the viewer files in `results/path_diagram/viewer/`; and the answer with each supernode switched off at "=". The diagram itself is drawn locally from these files.

Setup: Accelerator GPU T4, Internet on, secret `HF_TOKEN`. A few minutes; interactive is fine.

In [ ]:
!rm -rf helix_addition && git clone https://github.com/amark-23/helix_addition.git
!pip install -q git+https://github.com/safety-research/circuit-tracer@v0.5.3

In [ ]:
import os, sys
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # before torch loads
os.chdir("/kaggle/working/helix_addition")
sys.path.insert(0, "/kaggle/working/helix_addition/src")
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
from helixadd import model as hmodel, prompts, graphs, intervene
from helixadd.groups import load_labels, supernode
model = hmodel.load_model()

In [ ]:
import json, torch
import pandas as pd
from pathlib import Path
from circuit_tracer import attribute
from circuit_tracer.utils import create_graph_files

PAIRS = [(36, 59), (36, 53)]
OUT = Path("results/path_diagram")
for a, b in PAIRS:
    g = attribute(prompts.make_prompt(a, b), model, offload=None, verbose=False, batch_size=64)
    print(a, b, "kept nodes, edges:", graphs.save_pruned(g, OUT / f"{a}_{b}"))
    create_graph_files(g, slug=f"add-{a}-{b}", output_path=str(OUT / "viewer"))
    del g; torch.cuda.empty_cache()
json.dump({"pairs": PAIRS, "node_threshold": 0.8, "edge_threshold": 0.98, "batch_size": 64,
           "circuit_tracer": "v0.5.3"}, open(OUT / "params.json", "w"), indent=1)

In [ ]:
# answer with each supernode switched off at "=" (the same groups as Q1)
labels = load_labels()
rows = []
for name in ["none"] + graphs.GROUPS:
    family = "mixed" if name == "mixed family" else name
    group = supernode(labels, family) if name != "none" else supernode(labels, "last digit").iloc[:0]
    texts, _ = intervene.suppress_generate(model, [prompts.make_prompt(a, b) for a, b in PAIRS], group)
    rows += [{"a": a, "b": b, "switched off": name, "output": t} for (a, b), t in zip(PAIRS, texts)]
sup = pd.DataFrame(rows)
sup.to_csv(OUT / "suppression.csv", index=False)
sup.pivot(index="switched off", columns="b", values="output")

Then download `results/path_diagram/` (Output tab, Download All) and put it in the repo.